## 4 Preparação dos Dados e Feature Engineering

### 4.1 Imports e Configuração


In [2]:

import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, 
                             recall_score, f1_score, classification_report, 
                             confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.inspection import permutation_importance

print('✓ Imports concluídos')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

✓ Imports concluídos
Timestamp: 2025-11-26 20:52:12


In [ ]:
# Preparar dados (usa `df` já presente neste notebook)
print("="*80)
print("PREPARAÇÃO DOS DADOS PARA MODELAGEM")
print("="*80)

if 'df' not in globals():
    # fallback: carregar csv limpo se df não existir em memória
    cleaned = '/home/exati/Ciecia_dados_final/creditcard_cleaned.csv'
    if os.path.exists(cleaned):
        df = pd.read_csv(cleaned)
        print(f'✓ Carregado dataframe de: {cleaned}')
    else:
        raise RuntimeError('Dataframe `df` não encontrado e creditcard_cleaned.csv ausente.')

# Definir alvo e features
target = 'Class'
exclude_cols = ['Time', target] if 'Time' in df.columns else [target]
features = [c for c in df.columns if c not in exclude_cols and c != 'index']

X = df[features].copy()
y = df[target].astype(int)

print(f"\nFeatures selecionadas: {len(features)}")
print(f"Target: {target}")

# One-hot encoding para TimeOfDay se existir
if 'TimeOfDay' in X.columns:
    X = pd.get_dummies(X, columns=['TimeOfDay'], drop_first=True)
    print(f"✓ One-hot encoding aplicado em 'TimeOfDay'")

# Garantir valores numéricos e preencher NA
X = X.fillna(0)
X = X.select_dtypes(include=[np.number])

print(f"\nDimensões finais X: {X.shape}")
print(f"Distribuição de classes:")
print(f"  - Legítimas (0): {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")
print(f"  - Fraudes (1): {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.2f}%)")

# Split estratificado (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n✓ Split estratificado realizado:")
print(f"  - Train: {X_train.shape[0]:,} amostras")
print(f"  - Test:  {X_test.shape[0]:,} amostras")
print(f"\nDistribuição train: Legítimas={np.bincount(y_train)[0]:,}, Fraudes={np.bincount(y_train)[1]:,}")
print(f"Distribuição test:  Legítimas={np.bincount(y_test)[0]:,}, Fraudes={np.bincount(y_test)[1]:,}")